In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical

In [ ]:
CLASS_NAMES = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

(X_train, y_train_raw), (X_test, y_test_raw) = fashion_mnist.load_data()
print('Train:', X_train.shape, '| Test:', X_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for cls, ax in enumerate(axes.flatten()):
    idx = np.where(y_train_raw == cls)[0][0]
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(CLASS_NAMES[cls]); ax.axis('off')
plt.suptitle('Sample Images'); plt.tight_layout(); plt.show()

In [ ]:
X_train = X_train.reshape(-1,28,28,1).astype('float32') / 255.0
X_test  = X_test.reshape(-1,28,28,1).astype('float32')  / 255.0
y_train = to_categorical(y_train_raw, 10)
y_test  = to_categorical(y_test_raw,  10)

In [ ]:
model = Sequential([
    Conv2D(32,(3,3), activation='relu', input_shape=(28,28,1)),
    MaxPooling2D(2,2),
    Conv2D(64,(3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {acc*100:.2f}%')

y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test_raw, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_test_raw, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
indices = np.random.choice(len(X_test), 10, replace=False)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, idx in zip(axes.flatten(), indices):
    ax.imshow(X_test[idx].reshape(28,28), cmap='gray')
    p = CLASS_NAMES[y_pred[idx]]
    t = CLASS_NAMES[y_test_raw[idx]]
    ax.set_title(f'P:{p}\nT:{t}', color='green' if p==t else 'red', fontsize=8)
    ax.axis('off')
plt.suptitle('Predictions'); plt.tight_layout(); plt.show()